# 05 — Export to CrewAI

Take a generated topology and export it as a CrewAI crew config.

**Prerequisites:**
- Python 3.10+ with numpy
- Running from inside the `flybrain-connectome` checkout
- Completed [01_generate_visualize.ipynb](01_generate_visualize.ipynb)

## Cell 1 — Load a previously generated topology

In [ ]:
import sys, os, json
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]
TOOLKIT = HERE / "flybrain-connectome-mcp"
for p in [TOOLKIT, TOOLKIT / "tools"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Load the topology
org_dir = TOOLKIT / "generated_organisms"
json_path = org_dir / "demo_predictive_coding_v2_m5_a8.json"
if not json_path.exists():
    print("[warn] Run 'flybrain demo' first to generate a topology")
    import subprocess
    subprocess.run([sys.executable, str(HERE / "flybrain_cli.py"), "demo", "--no-open"],
                   cwd=str(HERE), check=True)

with open(json_path, "r", encoding="utf-8") as f:
    topo = json.load(f)

nodes = topo.get("organism", {}).get("nodes", [])
edges = topo.get("organism", {}).get("edges", [])
print(f"[loaded] {len(nodes)} nodes, {len(edges)} edges")

## Cell 2 — Export to CrewAI format

In [ ]:
# Use the export command
import subprocess

result = subprocess.run(
    [sys.executable, str(HERE / "flybrain_cli.py"), "export", str(json_path),
     "--format", "crewai", "--output", str(json_path.parent / "crewai_config.json")],
    capture_output=True, text=True, cwd=str(HERE)
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## Cell 3 — Inspect the CrewAI config

In [ ]:
with open(json_path.parent / "crewai_config.json", "r", encoding="utf-8") as f:
    crewai = json.load(f)

print("CrewAI config structure:")
print(f"  blueprint : {crewai['blueprint']}")
print(f"  format    : {crewai['format']}")
print(f"  agents    : {len(crewai['crew']['agents'])}")
print(f"  tasks     : {len(crewai['crew']['tasks'])}")
print(f"  process   : {crewai['crew']['process']}")
print()

# Show first 3 agents
print("First 3 agents:")
for a in crewai['crew']['agents'][:3]:
    print(f"  role: {a['role']}")
    print(f"    goal: {a['goal'][:80]}...")
    print(f"    module: {a['module']}")

## Cell 4 — Use with CrewAI (pseudocode)

In [ ]:
# Pseudocode for CrewAI integration (requires crewai installed)
# The config maps directly to CrewAI Crew + Agent + Task.

# from crewai import Crew, Agent, Task
# from crewai.process import SequentialProcess

# # Create agents
# agents = []
# for agent_def in crewai['crew']['agents']:
#     agent = Agent(
#         role=agent_def['role'],
#         goal=agent_def['goal'],
#         backstory=agent_def['backstory'],
#         verbose=True,
#     )
#     agents.append(agent)

# # Create tasks
# tasks = []
# for task_def in crewai['crew']['tasks']:
#     task = Task(
#         name=task_def['name'],
#         description=task_def['description'],
#         expected_output=task_def['expected_output'],
#         agent=task_def['agent'],
#     )
#     tasks.append(task)

# # Create crew
# crew = Crew(
#     agents=agents,
#     tasks=tasks,
#     process=SequentialProcess(),
#     verbose=True,
# )

# # Run
# result = crew.kickoff()

print("CrewAI integration ready!")
print(f"  Agents to create: {len(crewai['crew']['agents'])}")
print(f"  Tasks to define : {len(crewai['crew']['tasks'])}")

## Key insight

CrewAI is great for role-playing agent crews. The flybrain topology
gives you a **biologically-grounded** crew:

- Each agent = a neural role (sensory, motor, associative, etc.)
- Each task = a brain region's function (processing, integration, output)
- Each module = a brain area (cerebrum, cerebellum, etc.)

This is not just a list of agents — it's a **provably-traceable**
agent crew that can be audited with `flybrain audit`.

## Next

- [06_export_autogen.ipynb](06_export_autogen.ipynb) — export to AutoGen format